In [1]:
"""
既然神经网络里面有大量的参数是矩阵,那我们能不能直接从矩阵更新的角度设计优化器
    1. 先像momentum 一样维护一个动量方向
    2. 再对这个矩阵的更新方向进行正交化处理
"""

from collections.abc import Iterable
from typing import override

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

import dnnlpy
import dnnlpy.optim as dopt
import matplotlib.pyplot as plt

print('PyTorch Version:', torch.__version__)

PyTorch Version: 2.13.0+cpu


In [4]:
"""
Muon的PyTorch实现
"""


def newton_schulz_5(X: Tensor, ns_steps: int = 5,
                    ns_coefficients: tuple[float, float, float] = (3.4445, -4.7750, 2.0315),
                    eps: float = 1e-7) -> Tensor:
    if X.ndim != 2:
        raise ValueError("Moon  only supports 2-dimensional input")
    a, b, c = ns_coefficients
    X = X / (X.norm() + eps)
    should_transpose = X.size(0) > X.size(1)
    if should_transpose:
        X = X.T
    for _ in range(ns_steps):
        A = torch.mm(X, X.T)
        B = torch.addmm(A, A, A, beta=b, alpha=c)
        X = torch.addmm(X, B, X, beta=a)
    if should_transpose:
        X = X.T
    return X


class Muon(dopt.Optimizer):
    def __init__(self, params: Iterable[Tensor], lr: float = 1e-3, weight_decay: float = 1e-2, momentum: float = 0.95,
                 nesterov: bool = True, ns_coefficients: tuple = (3.4445, -4.7750, 2.0315), ns_steps: int = 5,
                 eps: float = 1e-7):
        super().__init__(params)
        self.lr = lr
        self.weight_decay = weight_decay
        self.momentum = momentum
        self.nesterov = nesterov
        self.ns_coefficients = ns_coefficients
        self.ns_steps = ns_steps
        self.eps = eps
        self.momentum_buffers = [torch.zeros_like(p) for p in params]

    @override
    @torch.no_grad()
    def step(self):
        for p, buffer in zip(self.params, self.momentum_buffers, strict=True):
            if p.grad is None:
                continue
            grad = p.grad
            if self.weight_decay > 0:
                p.mul_(1 - self.lr * self.weight_decay)
            buffer.mul_(self.momentum).add_(grad)
            if self.nesterov:
                direction = grad + self.momentum * buffer
            else:
                direction = buffer
            update = newton_schulz_5(direction, ns_steps=self.ns_steps, ns_coefficients=self.ns_coefficients,
                                     eps=self.eps)
            p.sub_(update, alpha=self.lr)


def loss_fn(theta: Tensor) -> Tensor:
    target = torch.tensor(
        [[2.0, 0.0], [0.0, -10]],
        device=theta.device,
        dtype=theta.dtype
    )
    scale = torch.tensor(
        [[0.1, 10.0], [10.0, 0.1]],
        device=theta.device,
        dtype=theta.dtype
    )
    return 0.5 * (scale * (theta - target) ** 2).sum()


theta = torch.tensor([[-5.0, 2.0], [1.0, 2.0]], requires_grad=True)
optimizer1 = dopt.SGD([theta], lr=0.1)
optimizer2 = Muon([theta], lr=0.2)

sgd_history = dopt.run_optimizer(optimizer1, loss_fn, steps=80)
muon_history = dopt.run_optimizer(optimizer2, loss_fn, steps=80)

print('SGD history:')
print('Final theta:', sgd_history[-1])
print('Final loss:', loss_fn(sgd_history[-1]).item())

print('Muon history:')
print('Final theta:', muon_history[-1])
print('Final loss:', loss_fn(muon_history[-1]).item())



SGD history:
Final theta: tensor([[-1.1327,  0.0000],
        [ 0.0000, -4.6297]])
Final loss: 1.9326730966567993
Muon history:
Final theta: tensor([[ 1.7630,  0.0000],
        [ 0.0000, -9.0585]])
Final loss: 0.047133177518844604
